In [1]:
import os

In [2]:
%pwd

'/home/muskan/PrOjects/Text-Summarizer/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/muskan/PrOjects/Text-Summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )
        return data_ingestion_config

In [8]:
import os
import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-29 10:07:27,455: INFO: yaml file: config/config.yaml loaded successfully]
[2026-03-29 10:07:27,460: INFO: yaml file: params.yaml loaded successfully]
[2026-03-29 10:07:27,463: INFO: created directory at: artifacts]
[2026-03-29 10:07:27,465: INFO: created directory at: artifacts/data_ingestion]
[2026-03-29 10:07:30,593: INFO: artifacts/data_ingestion/dat.zip download! with following info: 
Connection: close
Content-Length: 3596916
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "2b513bf5cd4a69ae70b97fc2e5c6f9adb5a61d559c881ffd3528b1076e0f52d7"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: E9CC:33CFFA:A1583D:B731ED:69C8AC88
Accept-Ranges: bytes
Date: Sun, 29 Mar 2026 04:37:29 GMT
Via: 1.1 varnish
X-Served-By: cache-del-vibw2260034-DEL
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S17747590